In [11]:
order = 1
reffe = ReferenceFE(lagrangian, VectorValue{2,Float64}, order)

V = TestFESpace(
  Ω_act, reffe;
  conformity = :H1
)

U = TrialFESpace(V)

UnconstrainedFESpace()

In [12]:
dΩ = Measure(Ω, 2*order)
dΓ = Measure(Γ, 2*order)

GenericMeasure()

In [13]:
E  = 210000.0                           # N/mm^2
ν  = 0.3
μ  = E/(2*(1+ν))
λ1 = (E*ν)/((1+ν)*(1-2*ν))

# plane stress correction
λ  = (2*λ1*μ)/(λ1+2*μ)   

# Traction
g = VectorValue(100.0,0.0)              # N

# Stress–strain relation
σ(ε) = λ*tr(ε)*one(ε) + 2*μ*ε
ε(u) = symmetric_gradient(u)

ε (generic function with 1 method)

In [14]:
t̄(x) = VectorValue(0.0, -1.0)

t̄ (generic function with 1 method)

In [15]:
a_bulk(u,v) = ∫( σ(u) ⊙ ε(v) )dΩ

a_bulk (generic function with 1 method)

In [16]:
n = get_normal_vector(Γ)
γ = 10.0*order^2

a_nitsche(u,v) =
  ∫( - (σ(u) ⋅ n) ⋅ v
     - (σ(v) ⋅ n) ⋅ u
     + γ*(u ⋅ v)
   )dΓ


a_nitsche (generic function with 1 method)

In [17]:
l_traction(v) = ∫( t̄ ⋅ v )dΓ

l_traction (generic function with 1 method)

In [18]:
Fh = Skeleton(Ω_act)

γg = 1e-2
a_ghost(u,v) =
  γg * ∫( jump(∇(u)) ⋅ jump(∇(v)) )dFh


a_ghost (generic function with 1 method)

In [19]:
a(u,v) = a_bulk(u,v) + a_nitsche(u,v) + a_ghost(u,v)
l(v)   = l_traction(v)

l (generic function with 1 method)

In [ ]:
E  = 1.0
ν  = 0.3
λ  = E*ν/((1+ν)*(1-2ν))
μ  = E/(2*(1+ν))

const I2 = TensorValue(1.0, 0.0, 0.0, 1.0)

ε(u) = 0.5 * (∇(u) + ∇(u)')
σ(u) = λ * tr(ε(u)) * I2 + 2μ * ε(u)

σ (generic function with 1 method)

In [22]:
op = AffineFEOperator(a, l, U, V)
uh = solve(op)

LoadError: UndefVarError: `dFh` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:



# Boundary tags
labels = get_face_labeling(model)
add_tag_from_tags!(labels,"left",[1])   # x = 0
add_tag_from_tags!(labels,"right",[2])  # x = L

# -----------------------------
# FE spaces
# -----------------------------
reffe = ReferenceFE(lagrangian,VectorValue{2,Float64},1)

V = TestFESpace(
  model,reffe;
  conformity=:H1,
  dirichlet_tags=["left"]
)

U = TrialFESpace(V,[VectorValue(0.0,0.0)])

# -----------------------------
# Weak form (CutFEM + Nitsche)
# -----------------------------
γ = 10.0          # Nitsche penalty
h = L/nx

a(u,v) =
  ∫( σ(ε(u)) ⊙ ε(v) )dΩ +

  # Nitsche (Dirichlet, left boundary)
  ∫(
    - (σ(ε(u))*n_Γ)⋅v
    - (σ(ε(v))*n_Γ)⋅u
    + γ*(μ/h)*(u⋅v)
  )dΓ

l(v) =
  ∫( g⋅v )dΓ

# -----------------------------
# Solve
# -----------------------------
op = AffineFEOperator(a,l,U,V)
uh = solve(op)

# -----------------------------
# Output
# -----------------------------
writevtk(
  model,
  "cantilever_cutfem_2d",
  cellfields = [
    "u" => uh,
    "vonMises" => sqrt(3/2)*norm(dev(σ(ε(uh))))
  ]
)

end


LoadError: UndefVarError: `rectangle` not defined in `Main.CantileverCutFEM2D`
Suggestion: check for spelling errors or missing imports.

In [ ]:
const E  = 210000.0                     # N/mm^2
const ν  = 0.3
const μ  = E/(2*(1+ν))
const λ1 = (E*ν)/((1+ν)*(1-2*ν))

# plane stress correction
const λ  = (2*λ1*μ)/(λ1+2*μ)   

# Traction
g = VectorValue(100.0,0.0)              # N

# Stress–strain relation
σ(ε) = λ*tr(ε)*one(ε) + 2*μ*ε
ε(u) = symmetric_gradient(u)